In [ ]:
# ============================================================
# IMPROVED DCGAN FOR MNIST
# ============================================================

!pip install -q tensorflow imageio


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import glob
import imageio.v2 as imageio

from tensorflow.keras import layers
from IPython import display


print("TensorFlow version:", tf.__version__)


# ============================================================
# 2. LOAD MNIST DATASET
# ============================================================

(train_images, _), (_, _) = tf.keras.datasets.mnist.load_data()

# Add channel dimension
train_images = train_images.reshape(
    train_images.shape[0], 28, 28, 1
).astype("float32")

# Normalize to [-1, 1]
train_images = (train_images - 127.5) / 127.5


BUFFER_SIZE = 60000
BATCH_SIZE = 128

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices(train_images)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset:", train_images.shape)


# ============================================================
# 3. GENERATOR
# ============================================================

def make_generator_model():

    model = tf.keras.Sequential(name="Generator")

    # Random noise: 100
    model.add(
        layers.Dense(
            7 * 7 * 256,
            use_bias=False,
            input_shape=(100,),
            kernel_initializer="glorot_uniform"
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # 7 x 7 x 256
    model.add(
        layers.Reshape((7, 7, 256))
    )

    # 7 x 7 -> 7 x 7
    model.add(
        layers.Conv2DTranspose(
            128,
            kernel_size=5,
            strides=1,
            padding="same",
            use_bias=False
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # 7 x 7 -> 14 x 14
    model.add(
        layers.Conv2DTranspose(
            64,
            kernel_size=5,
            strides=2,
            padding="same",
            use_bias=False
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # 14 x 14 -> 28 x 28
    model.add(
        layers.Conv2DTranspose(
            32,
            kernel_size=5,
            strides=2,
            padding="same",
            use_bias=False
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Final image
    model.add(
        layers.Conv2D(
            1,
            kernel_size=5,
            padding="same",
            activation="tanh"
        )
    )

    return model


generator = make_generator_model()

print("\nGENERATOR")
generator.summary()


# ============================================================
# 4. DISCRIMINATOR
# ============================================================

def make_discriminator_model():

    model = tf.keras.Sequential(name="Discriminator")

    # 28 x 28 -> 14 x 14
    model.add(
        layers.Conv2D(
            64,
            kernel_size=5,
            strides=2,
            padding="same",
            input_shape=[28, 28, 1]
        )
    )

    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))

    # 14 x 14 -> 7 x 7
    model.add(
        layers.Conv2D(
            128,
            kernel_size=5,
            strides=2,
            padding="same"
        )
    )

    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))

    # Additional feature extraction
    model.add(
        layers.Conv2D(
            256,
            kernel_size=3,
            strides=1,
            padding="same"
        )
    )

    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.25))

    model.add(layers.Flatten())

    model.add(
        layers.Dense(1)
    )

    return model


discriminator = make_discriminator_model()

print("\nDISCRIMINATOR")
discriminator.summary()


# ============================================================
# 5. LOSS FUNCTION
# ============================================================

cross_entropy = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)


# Discriminator loss
def discriminator_loss(real_output, fake_output):

    # Label smoothing:
    # Real = 0.9 instead of 1.0
    real_labels = tf.ones_like(real_output) * 0.9

    real_loss = cross_entropy(
        real_labels,
        real_output
    )

    fake_loss = cross_entropy(
        tf.zeros_like(fake_output),
        fake_output
    )

    return real_loss + fake_loss


# Generator loss
def generator_loss(fake_output):

    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )


# ============================================================
# 6. OPTIMIZERS
# ============================================================

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)


# ============================================================
# 7. CHECKPOINT
# ============================================================

checkpoint_dir = "./training_checkpoints"

os.makedirs(
    checkpoint_dir,
    exist_ok=True
)

checkpoint_prefix = os.path.join(
    checkpoint_dir,
    "ckpt"
)

checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator
)


# ============================================================
# 8. FIXED RANDOM SEED
# ============================================================

noise_dim = 100
num_examples = 16

seed = tf.random.normal(
    [num_examples, noise_dim]
)


# ============================================================
# 9. TRAINING STEP
# ============================================================

@tf.function
def train_step(images):

    noise = tf.random.normal(
        [tf.shape(images)[0], noise_dim]
    )

    with tf.GradientTape() as gen_tape, \
         tf.GradientTape() as disc_tape:

        # Generate fake images
        generated_images = generator(
            noise,
            training=True
        )

        # Discriminator
        real_output = discriminator(
            images,
            training=True
        )

        fake_output = discriminator(
            generated_images,
            training=True
        )

        # Loss
        gen_loss = generator_loss(
            fake_output
        )

        disc_loss = discriminator_loss(
            real_output,
            fake_output
        )

    # Generator gradients
    generator_gradients = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    # Discriminator gradients
    discriminator_gradients = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    # Update Generator
    generator_optimizer.apply_gradients(
        zip(
            generator_gradients,
            generator.trainable_variables
        )
    )

    # Update Discriminator
    discriminator_optimizer.apply_gradients(
        zip(
            discriminator_gradients,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss


# ============================================================
# 10. GENERATE SAMPLE IMAGES
# ============================================================

def generate_and_save_images(
    model,
    epoch,
    test_input
):

    predictions = model(
        test_input,
        training=False
    )

    fig = plt.figure(
        figsize=(6, 6)
    )

    for i in range(16):

        plt.subplot(4, 4, i + 1)

        plt.imshow(
            predictions[i, :, :, 0] * 127.5 + 127.5,
            cmap="gray"
        )

        plt.axis("off")

    plt.suptitle(
        f"Generated Images - Epoch {epoch}"
    )

    filename = (
        f"image_at_epoch_{epoch:04d}.png"
    )

    plt.savefig(
        filename,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 11. TRAINING FUNCTION
# ============================================================

def train(dataset, epochs):

    for epoch in range(epochs):

        start = time.time()

        gen_loss_total = 0
        disc_loss_total = 0
        batches = 0

        for image_batch in dataset:

            gen_loss, disc_loss = train_step(
                image_batch
            )

            gen_loss_total += float(gen_loss)
            disc_loss_total += float(disc_loss)

            batches += 1

        # Generate images after every epoch
        generate_and_save_images(
            generator,
            epoch + 1,
            seed
        )

        # Save checkpoint
        if (epoch + 1) % 10 == 0:

            checkpoint.save(
                file_prefix=checkpoint_prefix
            )

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Generator Loss: {gen_loss_total / batches:.4f} | "
            f"Discriminator Loss: {disc_loss_total / batches:.4f} | "
            f"Time: {time.time() - start:.2f}s"
        )


# ============================================================
# 12. TRAIN MODEL
# ============================================================

# First test with 10 epochs.
# For better quality use 30-50 epochs.

EPOCHS = 30

print("\nStarting training...\n")

train(
    train_dataset,
    EPOCHS
)

print("\nTraining completed!")


# ============================================================
# 13. SHOW FINAL GENERATED IMAGES
# ============================================================

generate_and_save_images(
    generator,
    EPOCHS,
    seed
)

img = imageio.imread(
    f"image_at_epoch_{EPOCHS:04d}.png"
)

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Final Generated Images")
plt.show()


# ============================================================
# 14. CREATE GIF
# ============================================================

anim_file = "dcgan_mnist.gif"

filenames = sorted(
    glob.glob("image_at_epoch_*.png")
)

with imageio.get_writer(
    anim_file,
    mode="I",
    duration=0.5
) as writer:

    for filename in filenames:

        image = imageio.imread(
            filename
        )

        writer.append_data(
            image
        )

print(
    "\nGIF created successfully:",
    anim_file
)


# ============================================================
# 15. DISPLAY GIF
# ============================================================

display.display(
    display.Image(
        filename=anim_file
    )
)

TensorFlow version: 2.21.0
Dataset: (60000, 28, 28, 1)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(



GENERATOR


Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 12544)          │     1,254,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12544)          │        50,176 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_3              │ (None, 7, 7, 128)      │       819,200 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_6 (LeakyReLU)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_4              │ (None, 14, 14, 64)     │       204,800 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_7 (LeakyReLU)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_5              │ (None, 28, 28, 32)     │        51,200 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_8 (LeakyReLU)       │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 1)      │           801 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_9 (LeakyReLU)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 128)      │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_10 (LeakyReLU)      │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 7, 7, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_11 (LeakyReLU)      │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │        12,545 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 514,305 (1.96 MB)

 Trainable params: 514,305 (1.96 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...

